Base: https://www.youtube.com/watch?v=9SpwelHPUo8

In [1]:
#Imports
import cv2
import pygame
import numpy 
import mediapipe
import sys 
import random

pygame 2.5.2 (SDL 2.28.3, Python 3.10.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
#Constants
width, height = 1000, 700


In [3]:
#Uploads
screen = pygame.display.set_mode((width, height))
pygame.display.set_caption("River Cleanning")

In [4]:
class IntroWin(pygame.sprite.Sprite):
    def __init__(self, thing, x, y, speed):
        pygame.sprite.Sprite.__init__(self, self.containers)
        self.image = pygame.image.load(thing).convert_alpha()
        self.image = pygame.transform.scale(self.image, (random.randint(70,130), random.randint(70,130)))
        
        self.rect = self.image.get_rect()
        self.rect.x = x
        self.rect.y = y
        
        self.speed = speed
    
    def update(self):
        screen.blit(self.image, (self.rect))
        
        if self.rect.y <= 0:
            self.kill()
        elif self.rect.y <= height:
            self.rect.y -= self.speed


In [5]:
def intro_window(background, object_path):
    pygame.init()
    font = pygame.font.Font(None, 70)

    running = True
    game_over = False

    max_thing_on_the_window = 10

    next_thing_batch = 5

    pygame.time.set_timer(next_thing_batch, 2000)

    all_things = pygame.sprite.Group()
    IntroWinSprite= pygame.sprite.Group()
    IntroWin.containers = IntroWinSprite, all_things

    background_image = pygame.image.load(background).convert_alpha()
    background_image = pygame.transform.scale(background_image, (width, height))
    background_image = pygame.transform.flip(background_image, True, False)

    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
                #degug
                #pygame.quit()
                #sys.exit()

            elif event.type == pygame.KEYDOWN:
                running = False
            
            elif event.type == next_thing_batch:
                 for i in range(random.randint(2,5)):
                     IntroWin(random.choice(object_path), random.randint(0, width), height, random.randint(1,3))
                                     
                
                
        if len(all_things) < max_thing_on_the_window:
            for i in range(random.randint(2,5)):
                IntroWin(random.choice(object_path), random.randint(0, width), height, random.randint(1,3))
        
        screen.blit(background_image, (0,0))
        start_text = font.render("Enter Any Key To Start The Game",True, (0,0,255), None)
        screen.blit(start_text, (width//2-400,height//2))
        
        IntroWinSprite.update()
        pygame.display.flip()        


In [6]:
class MainWin(pygame.sprite.Sprite):
    def __init__(self, thing, x, y, speed):
        pygame.sprite.Sprite.__init__(self, self.containers)
        self.image = pygame.image.load(thing).convert_alpha()
        self.image = pygame.transform.scale(self.image, (random.randint(70,130), random.randint(70,130)))
        
        self.rect = self.image.get_rect()
        self.rect.x = x
        self.rect.y = y
        
        self.speed = speed

        self.radius = 25
    
    def update(self, score):
        flag = False
        screen.blit(self.image, (self.rect))
        
        if self.rect.y <= 0:
            self.speed = 0
            self.rect.y = 0
        elif self.rect.y <= height: # may reduse the and
            self.rect.y -= self.speed
            mouse_position = pygame.mouse.get_pos()
            mouse_click = pygame.mouse.get_pressed()
            
            if mouse_click[0] and  self.rect.collidepoint(*mouse_position):
                pop = pygame.image.load('fish.png').convert_alpha()
                pop = pygame.transform.scale(pop, (135,100))
                self.image = pop
                screen.blit(pop, (mouse_position))

                #self.rect.y -= 1
                flag = True
                
            if flag:
                if self.rect.x >= 0:
                    self.rect.x -= 1
                else:
                    score += 10
                    self.kill()
                    flag = False
                
        return score
                
    def collide (self, all_things):
        collections = pygame.sprite.spritecollide(self, all_things, False, pygame.sprite.collide_circle)
        for each in collections:
            if each.speed == 0:
                self.speed = 0
                


In [7]:
def main_window(background, object_path):
    pygame.init()
    font = pygame.font.Font(None,25)
    think_drop = 10
    pygame.time.set_timer(think_drop, 3000)
    
    high_score = 500
    score = 0
    
    game_over = False

    all_things = pygame.sprite.Group()
    all_sprites = pygame.sprite.Group()
    
    MainWin.containers = all_sprites, all_things

    background_image = pygame.image.load(background).convert_alpha()
    background_image = pygame.transform.scale(background_image, (width, height))
    background_image = pygame.transform.flip(background_image, True, False)

    intro_window(background, object_path)
    while not game_over:
        for event in pygame.event.get():
            if event.dict.get('key') == 27:
                game_over = True
            elif event.type == think_drop:
                for i in range(random.randint(2,6)):
                    MainWin(random.choice(object_path), random.randint(0, width), height, random.randint(1,2))
            elif score == high_score:
                game_over = True
        screen.blit(background_image, (0,0))
        score_text = font.render(f'Score: {score}', True, (0,0,255), None)
        screen.blit(score_text, (0,0))

        if not game_over:
            for thing in all_things:
                score = thing.update(score)
            for thing in all_things:
                all_things.remove(thing)
                thing.collide(all_things)
                all_things.add(thing)
        else:
            all_sprites.clear(screen, background_image)
            all_sprites.update(score)
        
        pygame.display.flip()




In [8]:
if __name__== '__main__':
    object_path = ["bottle.png", "bag.png"]
    main_window("river.png", object_path)


KeyboardInterrupt: 